<a href="https://colab.research.google.com/github/Doctordoom007/test/blob/p1/Document_Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Summarization Tool (AI-Powered)

Run the cells below **in order** (Shift+Enter). The first run will take a
minute or two because it downloads a small pretrained AI model. After that,
re-running the last cell with new text is fast.


## Step 1 — Install the required packages

In [1]:
!pip install -q transformers torch

## Step 2 — Write the summarizer tool to a file

In [9]:
%%writefile document_summarizer.py
"""
Document Summarization Tool (v2 - uses model/tokenizer directly,
avoids the transformers pipeline() task-name issue)
"""

import argparse
import os
import sys


def load_summarizer(model_name="sshleifer/distilbart-cnn-12-6"):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    print(f"Loading AI model '{model_name}' ... (first run downloads it, please wait)")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return tokenizer, model


def read_input_text(args):
    if args.file:
        if not os.path.isfile(args.file):
            print(f"Error: file not found -> {args.file}")
            sys.exit(1)
        with open(args.file, "r", encoding="utf-8") as f:
            return f.read()
    elif args.text:
        return args.text
    else:
        print("Paste/type the text to summarize. Press Ctrl+D (Ctrl+Z on Windows) when done:\n")
        return sys.stdin.read()


def chunk_text(text, max_words=500):
    words = text.split()
    if not words:
        return []
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]


def summarize_one_chunk(tokenizer, model, text, max_length=130, min_length=30):
    inputs = tokenizer([text], max_length=1024, truncation=True, return_tensors="pt")
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        early_stopping=True,
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


def summarize(tokenizer, model, text, max_length=130, min_length=30):
    chunks = chunk_text(text)
    summaries = []
    for i, chunk in enumerate(chunks, 1):
        print(f"Summarizing chunk {i}/{len(chunks)}...")
        summaries.append(summarize_one_chunk(tokenizer, model, chunk, max_length, min_length))

    combined = " ".join(summaries)
    if len(chunks) > 1:
        print("Combining chunk summaries into one final summary...")
        return summarize_one_chunk(tokenizer, model, combined, max_length, min_length)
    return combined


def main():
    parser = argparse.ArgumentParser(description="AI-powered Document Summarization Tool")
    parser.add_argument("-f", "--file", help="Path to a .txt file to summarize")
    parser.add_argument("-t", "--text", help="Text to summarize, given directly as a string")
    parser.add_argument("--max_length", type=int, default=130, help="Maximum length of the summary")
    parser.add_argument("--min_length", type=int, default=30, help="Minimum length of the summary")
    args = parser.parse_args()

    text = read_input_text(args)
    if not text.strip():
        print("No text provided.")
        sys.exit(1)

    tokenizer, model = load_summarizer()
    summary = summarize(tokenizer, model, text, args.max_length, args.min_length)

    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(summary)


if __name__ == "__main__":
    main()

Overwriting document_summarizer.py


## Step 3 — Try it on a sample paragraph

This uses a bit of built-in sample text so you can see it working right
away. Feel free to replace `sample_text` with your own paragraph.

In [10]:
sample_text = """
Artificial intelligence is transforming the way people work, learn, and communicate.
Businesses are using AI-powered tools to automate repetitive tasks, analyze large
amounts of data, and make faster decisions. In education, AI is being used to
personalize learning experiences for students based on their individual pace and
needs.
"""

with open("sample.txt", "w") as f:
    f.write(sample_text)

!python document_summarizer.py -f sample.txt


Loading AI model 'sshleifer/distilbart-cnn-12-6' ... (first run downloads it, please wait)
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 97.5kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 24.1MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 74.7MB/s]
pytorch_model.bin: 100% 1.22G/1.22G [00:12<00:00, 96.8MB/s]
model.safetensors:   0% 0.00/1.22G [00:00<?, ?B/s][transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
model.safetensors:  23% 277M/1.22G [00:03<00:07, 131MB/s]
Loading weights:   0% 0/358 [00:00<?, ?it/s]
Loading weights:  28% 102/358 [00:00<00:00, 989.54it/s]
Loading weights: 100% 358/358 [00:00<00:00, 1239.88it/s]
model.safetensors:  35% 429M/1.22G [00:04<00:05, 133MB/s]Summarizing chunk 1/1...
model.safetensors:  74% 907M/1.22G [00:10<00:03, 87.9MB/s]
SUMMARY
 Businesses are using AI-powered tools to automate repetitive tasks and analyze large amounts of data . In education, AI is being used to personalize learning experiences for studen

## Step 4 — Try it on your own text or file

**Option A: paste your own text below**

In [14]:
my_text = """
Paste your own paragraph or document text here, replacing this sample sentence.
"""

with open("my_document.txt", "w") as f:
    f.write(my_text)

!python document_summarizer.py -f my_document.txt


Loading AI model 'sshleifer/distilbart-cnn-12-6' ... (first run downloads it, please wait)
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100% 358/358 [00:00<00:00, 28364.52it/s]
Summarizing chunk 1/1...

SUMMARY
 Paste your own paragraph or document text here, replacing it with a sample sentence . Use this sample sentence to help students with reading comprehension and vocabulary .


**Option B: upload a .txt file** (run this cell, click "Choose Files", then
run the cell after it)

In [15]:
from google.colab import files
uploaded = files.upload()


In [13]:
# Replace 'yourfile.txt' with the filename you just uploaded
!python document_summarizer.py -f demo_document.txt


Loading AI model 'sshleifer/distilbart-cnn-12-6' ... (first run downloads it, please wait)
[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100% 358/358 [00:00<00:00, 18804.53it/s]
Summarizing chunk 1/1...

SUMMARY
 Businesses are increasingly using AI-powered tools to automate repetitive tasks, analyze large amounts of data, and make faster and more informed decisions . Many jobs that involve repetitive or predictable tasks are at risk of being automated, which could lead to significant shifts in the labor market . Despite these advances, experts are raising concerns about job displacement, data privacy and ethical use of AI systems .


## How this works (for your presentation)

- **The AI model:** a pretrained transformer neural network called **BART**
  (`distilbart-cnn-12-6`), downloaded once from Hugging Face and then run
  locally in this notebook - no API key, no ongoing internet calls.
- **Abstractive summarization:** the model reads the text and writes a new,
  shorter version in its own words, rather than just copying sentences.
- **Chunking:** long documents are split into ~500-word pieces first, because
  the AI model can only process a limited amount of text at once. Each piece
  is summarized, and if there was more than one piece, the results are
  combined and summarized once more into a single final summary.
